# Script 08 — Histograms and reports

Generates per-flight histograms and statistics, then creates PDF summaries for vegetation-index histograms and canopy temperature.

In [ ]:
# Imports
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import rasterio

from rasterio.mask import mask
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm


In [ ]:
# Project paths
# This notebook is expected to run from the notebooks/ directory.
ROOT_DIR = Path("..")

# Input data
INDICES_DIR = ROOT_DIR / "indices_aligned"
CROP_BOUNDARY = ROOT_DIR / "aoi" / "crop_boundary.gpkg"

# Histogram output
OUTPUT_DIR = ROOT_DIR / "histograms"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PDF report outputs
HISTOGRAM_REPORTS_DIR = ROOT_DIR / "indices_reports"
HISTOGRAM_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TEMPERATURE_REPORTS_DIR = ROOT_DIR / "temperature_reports"
TEMPERATURE_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Paths ready.")


In [ ]:
# Load crop boundary
crop_aoi = gpd.read_file(CROP_BOUNDARY)

print(crop_aoi)
print("\nCrop boundary loaded.")


In [ ]:
# Extract valid raster pixels inside crop_boundary
def extract_valid_pixels(raster_path):
    with rasterio.open(raster_path) as src:
        crop = crop_aoi.to_crs(src.crs) if crop_aoi.crs != src.crs else crop_aoi

        clipped, _ = mask(src, crop.geometry, crop=True)
        values = clipped[0].astype(np.float32)

        if src.nodata is not None:
            values = values[values != src.nodata]

        values = values[np.isfinite(values)]
        name = raster_path.name.lower()

        # Plausible ranges for vegetation indices and canopy temperature
        if "canopy_temperature" not in name:
            values = values[(values >= -1) & (values <= 1)]
        else:
            values = values[(values > -50) & (values < 80)]

        return values


In [ ]:
#Main function
def generate_histograms():

    year_folders = sorted([
        f for f in INDICES_DIR.iterdir()
        if f.is_dir()
    ])

    for year_folder in year_folders:

        year_name = year_folder.name

        print(f"\nPROCESSING YEAR: {year_name}")

        output_year = OUTPUT_DIR / year_name
        output_year.mkdir(parents=True, exist_ok=True)

        flight_folders = sorted([
            f for f in year_folder.iterdir()
            if f.is_dir()
        ])

        for flight_folder in tqdm(flight_folders):

            flight_name = flight_folder.name

            output_flight = output_year / flight_name
            output_flight.mkdir(parents=True, exist_ok=True)

            # Skip already processed flights
            existing_pngs = list(
                output_flight.glob("*.png")
            )

            if (
                len(existing_pngs) >= 4 and
                (output_flight / "statistics.csv").exists()
            ):

                print(
                    f"{flight_name} already processed. Skipping."
                )

                continue

            statistics = []

            tif_files = sorted(
                flight_folder.glob("*.tif")
            )

            for tif_path in tif_files:

                values = extract_valid_pixels(
                    tif_path
                )

                if len(values) == 0:

                    continue

                filename = tif_path.stem.lower()

                # -------------------------
                # Identify raster
                # -------------------------

                if "gndvi" in filename:

                    index_name = "GNDVI"

                elif "ndvi" in filename:

                    index_name = "NDVI"

                elif "ndre" in filename:

                    index_name = "NDRE"

                elif "canopy_temperature" in filename:

                    index_name = "Canopy Temperature"

                else:

                    continue

                # -------------------------
                # Statistics
                # -------------------------

                p5 = np.percentile(values, 5)
                p25 = np.percentile(values, 25)
                p75 = np.percentile(values, 75)
                p95 = np.percentile(values, 95)

                statistics.append({

                    "Index": index_name,
                    "Std": np.std(values),
                    "Min": np.min(values),
                    "Max": np.max(values),
                    "P5": p5,
                    "P25": p25,
                    "P75": p75,
                    "P95": p95

                })

                # -------------------------
                # Histogram
                # -------------------------

                plt.figure(figsize=(9, 6))

                plt.hist(
                    values,
                    bins=50,
                    edgecolor="black",
                    alpha=0.75
                )

                # P5
                plt.axvline(
                    p5,
                    color="red",
                    linestyle="--",
                    linewidth=2,
                    label=f"P5 = {p5:.3f}"
                )

                # P25
                plt.axvline(
                    p25,
                    color="orange",
                    linestyle="--",
                    linewidth=2,
                    label=f"P25 = {p25:.3f}"
                )

                # P75
                plt.axvline(
                    p75,
                    color="orange",
                    linestyle="--",
                    linewidth=2,
                    label=f"P75 = {p75:.3f}"
                )

                # P95
                plt.axvline(
                    p95,
                    color="red",
                    linestyle="--",
                    linewidth=2,
                    label=f"P95 = {p95:.3f}"
                )

                plt.title(
                    f"{index_name}\n{flight_name}"
                )

                plt.xlabel(index_name)

                plt.ylabel("Pixel Count")

                plt.grid(True)

                plt.legend()

                plt.tight_layout()

                plt.savefig(
                    output_flight /
                    f"{index_name}_histogram.png",
                    dpi=300,
                    bbox_inches="tight"
                )

                plt.close()

            # -------------------------
            # Save statistics
            # -------------------------

            df_stats = pd.DataFrame(
                statistics
            )

            df_stats.to_csv(
                output_flight /
                "statistics.csv",
                index=False
            )

            print(
                f"Saved statistics for {flight_name}"
            )

    print("\nALL HISTOGRAMS GENERATED.")

In [ ]:
generate_histograms()

## PDF report: vegetation-index histograms

In [ ]:

# Create PDF summaries of the generated histograms
HISTOGRAMS_DIR = ROOT_DIR / "histograms"

def create_histogram_report(year):

    year_folder = HISTOGRAMS_DIR / str(year)

    flight_folders = sorted([
        f for f in year_folder.iterdir()
        if f.is_dir()
    ])

    output_pdf = (
        HISTOGRAM_REPORTS_DIR /
        f"histograms_summary_{year}.pdf"
    )

    with PdfPages(output_pdf) as pdf:

        flights_per_page = 3

        for start in range(0, len(flight_folders), flights_per_page):

            subset = flight_folders[
                start:start + flights_per_page
            ]

            fig, axes = plt.subplots(
                len(subset),
                3,
                figsize=(11.69, 8.27)
            )

            if len(subset) == 1:
                axes = [axes]

            for row, flight in enumerate(subset):

                date = flight.name.replace(
                    "flight_", ""
                )

                files = {
                    "GNDVI": flight / "GNDVI_histogram.png",
                    "NDVI": flight / "NDVI_histogram.png",
                    "NDRE": flight / "NDRE_histogram.png"
                }

                for col, (label, path) in enumerate(files.items()):

                    ax = axes[row][col]

                    if path.exists():

                        img = mpimg.imread(path)

                        ax.imshow(img)

                    ax.set_xticks([])
                    ax.set_yticks([])

                    ax.set_title(
                        f"{label}\n{date}",
                        fontsize=11
                    )

            plt.tight_layout()

            pdf.savefig(
                fig,
                bbox_inches="tight"
            )

            plt.close(fig)

    print(
        f"{year} report saved -> {output_pdf}"
    )



In [ ]:
create_histogram_report(2025)
create_histogram_report(2026)

## PDF report: canopy temperature

In [ ]:
# Create PDF reports for canopy temperature maps and histograms
INDICES_DIR = ROOT_DIR / "indices_aligned"
HISTOGRAMS_DIR = ROOT_DIR / "histograms"

def create_temperature_report(year):

    year_indices = INDICES_DIR / str(year)
    year_histograms = HISTOGRAMS_DIR / str(year)

    flights = sorted([
        f for f in year_indices.iterdir()
        if f.is_dir()
    ])

    output_pdf = (
        TEMPERATURE_REPORTS_DIR /
        f"canopy_temperature_report_{year}.pdf"
    )

    with PdfPages(output_pdf) as pdf:

        flights_per_page = 2

        for start in range(0, len(flights), flights_per_page):

            subset = flights[start:start + flights_per_page]

            fig, axes = plt.subplots(
                len(subset),
                2,
                figsize=(11.69, 8.27)
            )

            if len(subset) == 1:
                axes = [axes]

            for row, flight in enumerate(subset):

                date = flight.name.replace(
                    "flight_", ""
                )

                thermal_path = (
                    flight /
                    f"{date}_canopy_temperature.tif"
                )

                histogram_path = (
                    year_histograms /
                    flight.name /
                    "Canopy Temperature_histogram.png"
                )

                if (
                    not thermal_path.exists()
                    or
                    not histogram_path.exists()
                ):
                    continue

                # =====================================================
                # CLIP THERMAL MAP TO crop_boundary
                # =====================================================

                with rasterio.open(thermal_path) as src:

                    if crop_aoi.crs != src.crs:
                        crop = crop_aoi.to_crs(src.crs)
                    else:
                        crop = crop_aoi

                    clipped, _ = mask(
                        src,
                        crop.geometry,
                        crop=True
                    )

                    image = clipped[0].astype(np.float32)

                    if src.nodata is not None:
                        image[
                            image == src.nodata
                        ] = np.nan

                    image[
                        ~np.isfinite(image)
                    ] = np.nan

                # =====================================================
                # LEFT: TEMPERATURE MAP
                # =====================================================

                im = axes[row][0].imshow(
                    image,
                    cmap="inferno"
                )

                axes[row][0].set_title(
                    f"{date}\nCanopy Temperature",
                    fontsize=12
                )

                axes[row][0].axis("off")

                plt.colorbar(
                    im,
                    ax=axes[row][0],
                    shrink=0.75,
                    label="Temperature"
                )

                # =====================================================
                # RIGHT: HISTOGRAM
                # =====================================================

                hist = mpimg.imread(
                    histogram_path
                )

                axes[row][1].imshow(hist)

                axes[row][1].set_title(
                    f"{date}\nHistogram",
                    fontsize=12
                )

                axes[row][1].axis("off")

            plt.tight_layout()

            pdf.savefig(
                fig,
                bbox_inches="tight"
            )

            plt.close(fig)

    print(f"Saved: {output_pdf}")



In [ ]:
create_temperature_report(2025)
create_temperature_report(2026)